# Phase 6 · AI 학습의 정체 — 흉내내기의 수학 🏋️

웹 챕터 **"AI 학습의 정체"** 와 짝을 이루는 Python 실습입니다.
웹 훈련소에서 본 것을, 여기서는 **진짜 경사하강 코드로 재현**하고 —
한 발 더 나아가 **분포표를 본 적 없는 모델이 샘플만으로 세상을 복원**하는 걸 봅니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터** — 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요 — 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch6')

## 1. 흉내내기 = 경사하강 — 웹 훈련소를 코드로

모델: 로짓 z 를 소프트맥스에 넣어 분포 q 를 내놓는 초미니 모델.
크로스 엔트로피 손실의 로짓 기울기는 놀랍게도 딱 **q − p** — 그래서 한 걸음은
`z ← z − lr·(q − p)` 입니다. (이 단순한 기울기가 4장에서 예고한 CE 의 선물!)

In [ ]:
import numpy as np

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(np.sum(p * np.log2(1 / p)))

def cross_entropy(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    return float(np.sum(p[mask] * np.log2(1 / q[mask])))

tokens = ['밥', '빵', '국수', '커피']
p_data = np.array([1/2, 1/4, 1/8, 1/8])   # 세상: "나는 아침에 ___" 다음 단어 분포
H_floor = entropy(p_data)                    # 바닥 = 1.75비트

rng = np.random.default_rng(42)
z = rng.normal(0, 2, size=4)                 # 엉터리 모델에서 출발
lr = 0.5

losses = []
for step in range(120):
    q = softmax(z)
    losses.append(cross_entropy(p_data, q))
    z = z - lr * (q - p_data)                # 경사하강 한 걸음 (∂CE/∂z = q − p)

q_final = softmax(z)
print('세상 p      :', np.round(p_data, 4))
print('시작 모델 q :', np.round(softmax(rng.normal(0, 2, size=4)), 4), '(예시 — 엉터리)')
print('훈련 후 q   :', np.round(q_final, 4))
print(f'\n최종 손실 {losses[-1]:.4f}비트 vs 바닥 H(p) = {H_floor}비트')
assert cross_entropy(p_data, q_final) - H_floor < 1e-4, '바닥까지 못 내려갔다?'
print('✅ 모델이 세상을 흉내내는 데 성공 — 초과분(KL)이 0으로 죽었습니다.')

## 2. 손실 곡선 — 0이 아니라 바닥에서 멈춘다

훈련 로그에서 늘 보는 그 미끄럼틀 곡선입니다. 어디서 평평해지는지 보세요.

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(losses, color='#6366f1', lw=2.5, label='훈련 손실 H(p, q)')
ax.axhline(H_floor, color='#ef4444', lw=2, linestyle='--',
           label=f'바닥 H(p) = {H_floor}비트 — 세상의 엔트로피')
ax.set_xlabel('경사하강 걸음 수')
ax.set_ylabel('손실 (비트)')
ax.set_title('학습 = 초과분(KL)을 깎는 일 — 손실은 0이 아니라 바닥에서 멈춘다')
ax.grid(True, color='#eef2f7')
ax.legend()
plt.show()

## 3. 진짜 학습 상황 — 분포표 없이 샘플만으로

1절은 사실 반칙이었어요 — 기울기 계산에 p_data 를 그대로 썼으니까요.
실제 세상에서는 **분포표를 아무도 못 봅니다**. 모델이 보는 건 뽑힌 샘플(훈련 데이터)뿐.

샘플 하나 = 원핫 정답이므로 기울기는 `q − onehot(정답)`, 실전처럼 100개씩 묶어
평균낸 기울기로 걷습니다(**미니배치 SGD** — 실제 딥러닝 훈련 방식 그대로).
이것만으로 세상이 복원될까요?

In [ ]:
samples = rng.choice(4, size=4000, p=p_data)   # 세상에서 뽑힌 훈련 데이터 4000개

z = rng.normal(0, 2, size=4)                   # 다시 엉터리 모델에서 출발
n_epochs = 30
for epoch in range(n_epochs):                  # 데이터를 100개씩 묶어 여러 번 반복 (미니배치 SGD)
    lr = 0.5 if epoch < 20 else 0.05           # 마지막엔 걸음을 줄여 정착 (학습률 스케줄)
    for start in range(0, len(samples), 100):
        batch = samples[start:start + 100]
        q = softmax(z)
        onehot_mean = np.bincount(batch, minlength=4) / len(batch)   # 배치 정답들의 평균 원핫
        z = z - lr * (q - onehot_mean)         # 기울기 = q − (배치의 원핫 평균)

q_learned = softmax(z)
freq = np.bincount(samples, minlength=4) / len(samples)

print('토큰   | 세상 p  | 샘플 빈도 | 학습된 q')
print('-' * 44)
for i, t in enumerate(tokens):
    print(f'{t:<4s} | {p_data[i]:.4f} | {freq[i]:.4f}   | {q_learned[i]:.4f}')

print('\n모델은 분포표를 본 적이 없는데, 원핫 샘플 4000개만으로 세상을 복원했습니다.')
print('4장의 "토큰마다 −log q(정답) 평균" 학습이 정확히 이 과정이에요.')
assert np.abs(q_learned - freq).max() < 0.02, '샘플 빈도조차 못 배웠다?'

## 4. 최대우도와 같은 문제 — 숫자로 확인

통계 교과서의 **최대우도**(데이터가 나올 확률을 최대화)와 우리의 CE 최소화가
같은 문제인지, 훈련 전/후 모델로 확인합니다.

In [ ]:
z_bad = rng.normal(0, 2, size=4)
q_bad = softmax(z_bad)

def avg_neg_log_q(q, xs):
    """샘플들의 −log₂ q 평균 = 훈련 손실 (경험적 크로스 엔트로피)."""
    return float(np.mean(-np.log2(q[xs])))

def log_likelihood(q, xs):
    """데이터 전체가 나올 로그 확률 log₂ Π q(xᵢ) = Σ log₂ q(xᵢ)."""
    return float(np.sum(np.log2(q[xs])))

print(f'{"모델":<10s} | 손실(−log q 평균) | 로그 우도(전체)')
print('-' * 52)
for name, qq in [('엉터리', q_bad), ('훈련됨', q_learned)]:
    print(f'{name:<10s} | {avg_neg_log_q(qq, samples):15.4f}비트 | {log_likelihood(qq, samples):12.0f}비트')

print('\n손실이 낮을수록 로그 우도가 높다 — 정확히 부호 반대·상수배 관계.')
print('통계학자의 "우도 최대화"와 정보이론가의 "초과 비트 최소화"는 같은 산입니다.')
assert avg_neg_log_q(q_learned, samples) < avg_neg_log_q(q_bad, samples)
assert log_likelihood(q_learned, samples) > log_likelihood(q_bad, samples)

## 5. 직접 풀어보기 ✏️

세상 분포가 p = (1/2, 1/4, 1/8, 1/8) 일 때, **완벽하게 배운 모델(q = p)의 훈련 손실**은
몇 비트일까요? (0이 아닙니다!)

In [ ]:
# TODO: 아래 answer 를 여러분의 계산으로 바꾸고 실행하세요.
# 힌트: 손실 = H(p) + D(p‖q) 인데 완벽히 배우면 D = 0 — cross_entropy(p_data, p_data) 로도 확인 가능

answer = cross_entropy(p_data, p_data)
print('내 답:', answer)
assert abs(answer - 1.75) < 1e-9, '다시 생각해 보세요 — 완벽한 모델에게 남는 것은 세상의 엔트로피'
print('✅ 잘했어요! 신도 1.75비트 아래로는 못 갑니다 — 그게 세상의 진짜 불확실성이니까요.')

---
**오늘의 한 줄**: AI 학습 = D(p_data‖q_θ) 최소화 = 크로스 엔트로피 최소화 =
세상 분포 흉내내기. 손실은 0이 아니라 세상의 엔트로피에서 멈춘다.

다음 챕터는 LLM 트랙 — 이렇게 배운 모델의 성적표를 사람이 읽기 좋은 숫자로 바꿉니다:
**퍼플렉시티 = 2^손실 = "평균 몇 갈래에서 고민하나"**. 🙂